# Лучшая конкретная модель и ансамбль — v21 / Run All

Выбираем лучшие конкретные веса, **не среднее по seed**. Готовые модели → пять изменений ансамбля → реранкинг финалистов. Новых optimizer updates нет; для двух усреднений пересчитывается BatchNorm на train.

Нажми **Restart Kernel → Run All** в прежнем исследовательском `.venv`. Интернет и другое устройство не нужны. Общего ограничения времени нет. Оставь компьютер на питании и без сна.

MVP, исходная validation/bbox, старые runs и релиз не меняются. Это поиск на уже изучаемых внутренних данных, не новая независимая оценка. Лучший результат не означает автоматическую смену MVP.


In [1]:
from pathlib import Path
import os, sys, json, subprocess
candidates = [Path.cwd(), Path.cwd() / 'Car-classification-MSK', *Path.cwd().parents]
REPO = next((p for p in candidates if (p / 'training/model_search.py').is_file()), None)
if REPO is None:
    raise RuntimeError('Открой notebook из исследовательского Car-classification-MSK')
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
import torch
from IPython.display import Markdown, display
from training import model_search as experiment
SOURCE_RUN = 'review_v1'
POLICY_RUN = 'policy_v1'
QUALITY_RUN = 'quality_v1'
RUN_NAME = 'search_v1'
source_manifest = json.loads((REPO / 'OSNet-AIN-x1.0/variant_16_review_protocol/runs' / SOURCE_RUN / 'manifest.json').read_text())
torch.set_num_threads(source_manifest['runtime']['torch_threads'])
torch.use_deterministic_algorithms(source_manifest['runtime']['deterministic'])
print('Repository:', REPO, '\nKernel:', sys.executable)
print('Run:', RUN_NAME, '| device:', source_manifest['runtime']['device'], '| optimizer updates: 0')


Repository: /Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK 
Kernel: /Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/.venv/bin/python
Run: search_v1 | device: mps | optimizer updates: 0


## 1. Быстрые проверки
Синтетические тесты: геометрия ансамбля, независимость query, сохранность источников, train-only BatchNorm, отбор конкретного seed, возобновление после ошибки.


In [2]:
subprocess.run([sys.executable, '-m', 'pytest', '-q', 'tests/test_model_search.py'], check=True)


...................                                                      [100%]
19 passed in 2.27s


CompletedProcess(args=['/Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/.venv/bin/python', '-m', 'pytest', '-q', 'tests/test_model_search.py'], returncode=0)

## 2. Проверка и фиксация источников
Читаются завершённые v16/v18/v20. Сохраняется отдельный manifest v21. Старые результаты не переписываются; runtime и разбиения должны совпадать.


In [3]:
context = experiment.prepare(RUN_NAME, source_run=SOURCE_RUN, policy_run=POLICY_RUN, quality_run=QUALITY_RUN)
print('Результаты:', context['output'])
print(json.dumps(context['manifest']['search_plan'], ensure_ascii=False, indent=2))


Результаты: /Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/OSNet-AIN-x1.0/variant_21_model_search/runs/search_v1
{
  "source_run": "review_v1",
  "policy_run": "policy_v1",
  "quality_run": "quality_v1",
  "components": {
    "R1_control_20260915": {
      "case": "R1_control",
      "seed": 20260915
    },
    "R1_control_20260916": {
      "case": "R1_control",
      "seed": 20260916
    },
    "R1_control_20260917": {
      "case": "R1_control",
      "seed": 20260917
    },
    "R1_full1700_20260915": {
      "case": "R1_full1700",
      "seed": 20260915
    },
    "R1_full1700_20260916": {
      "case": "R1_full1700",
      "seed": 20260916
    },
    "R1_full1700_20260917": {
      "case": "R1_full1700",
      "seed": 20260917
    },
    "R1_full1700_bn_20260915": {
      "case": "R1_full1700_bn",
      "seed": 20260915
    },
    "R1_full1700_bn_20260916": {
      "case": "R1_full1700_bn",
      "seed": 20260916
    },
    "R1_full1700_

## 3. Все сравнения
13 encoder, 6 готовых head и equal3; затем 5 ансамблей; затем λ=0.50/0.65/0.75/0.85 для финалистов. Выбор по среднему mAP трёх одинаковых эпизодов одной конкретной системы. Head применяется только к своему encoder.

После сбоя повтори **Restart Kernel → Run All** с тем же RUN_NAME. Готовые задачи проверяются и пропускаются. Не меняй код и не запускай два notebook одновременно.


In [4]:
results = experiment.run(context)


[START] features_R1_control_20260915 | elapsed 0.00 h
[DONE] features_R1_control_20260915 | 0.0 min
[START] features_R1_control_20260916 | elapsed 0.00 h
[DONE] features_R1_control_20260916 | 0.0 min
[START] features_R1_control_20260917 | elapsed 0.00 h
[DONE] features_R1_control_20260917 | 0.0 min
[START] features_R1_full1700_20260915 | elapsed 0.00 h
  R1_full1700_20260915: 32/1036 images
  R1_full1700_20260915: 352/1036 images
  R1_full1700_20260915: 672/1036 images
  R1_full1700_20260915: 992/1036 images
  R1_full1700_20260915: 1036/1036 images
[DONE] features_R1_full1700_20260915 | 0.3 min
[START] features_R1_full1700_20260916 | elapsed 0.01 h
  R1_full1700_20260916: 32/1036 images
  R1_full1700_20260916: 352/1036 images
  R1_full1700_20260916: 672/1036 images
  R1_full1700_20260916: 992/1036 images
  R1_full1700_20260916: 1036/1036 images
[DONE] features_R1_full1700_20260916 | 0.3 min
[START] features_R1_full1700_20260917 | elapsed 0.01 h
  R1_full1700_20260917: 32/1036 images
  

In [5]:
display(Markdown((context['output'] / 'REPORT.md').read_text()))
print('Точный кандидат:', context['output'] / 'selected_candidate.json')
print('Подробные результаты:', context['output'] / 'results.json')
print('Это внутренний выбор. Финального threshold, full-train, конкурсного экспорта и смены MVP не было.')


# v21 — выбор конкретной модели и ансамбля

Статус: **complete**.
Выбор по primary development; не новый независимый тест. Среднего gate по seed нет.
Optimizer updates: 0. Outer/alternate/test не оцениваются. MVP и исходная разметка не меняются.

| Конфигурация | λ | mAP@10 |
|---|---:|---:|
| replace_20260915_lambda65 | 0.65 | 0.852766 |
| replace_20260915_lambda50 | 0.50 | 0.852737 |
| replace_20260915_lambda75 | 0.75 | 0.852313 |
| replace_20260915_lambda85 | 0.85 | 0.852302 |
| R1_equal3_lambda75 | 0.75 | 0.852218 |
| add_w10_lambda75 | 0.75 | 0.852184 |
| add_w10_lambda85 | 0.85 | 0.851034 |
| R1_equal3_lambda85 | 0.85 | 0.850922 |
| replace_20260916_lambda75 | 0.75 | 0.850708 |
| add_w10_lambda65 | 0.65 | 0.850595 |
| R1_equal3_lambda65 | 0.65 | 0.850527 |
| add_w25_lambda75 | 0.75 | 0.849431 |
| head_w10_20260916_lambda75 | 0.75 | 0.849352 |
| R1_full1700_avg_20260915_lambda75 | 0.75 | 0.849319 |
| head_w05_20260916_lambda75 | 0.75 | 0.848728 |
| add_w10_lambda50 | 0.50 | 0.848641 |
| R1_full1700_bn_20260915_lambda75 | 0.75 | 0.848415 |
| replace_20260917_lambda75 | 0.75 | 0.848364 |
| R1_control_20260916_lambda75 | 0.75 | 0.848363 |
| R1_equal3_lambda50 | 0.50 | 0.846825 |
| R1_control_20260915_lambda75 | 0.75 | 0.844099 |
| head_w05_20260915_lambda75 | 0.75 | 0.843245 |
| head_w10_20260915_lambda75 | 0.75 | 0.842818 |
| R1_full1700_avg_20260916_lambda75 | 0.75 | 0.842717 |
| R1_full1700_20260916_lambda75 | 0.75 | 0.842613 |
| R1_full1700_bn_20260916_lambda75 | 0.75 | 0.841621 |
| R1_full1700_20260915_lambda75 | 0.75 | 0.840027 |
| R1_cosine800_20260915_lambda75 | 0.75 | 0.837527 |
| R1_control_20260917_lambda75 | 0.75 | 0.836375 |
| head_w05_20260917_lambda75 | 0.75 | 0.835971 |
| R1_full1700_20260917_lambda75 | 0.75 | 0.835325 |
| head_w10_20260917_lambda75 | 0.75 | 0.835324 |
| R1_full1700_bn_20260917_lambda75 | 0.75 | 0.824657 |
| R1_full1700_avg_20260917_lambda75 | 0.75 | 0.824096 |

Лучший внутренний кандидат: **replace_20260915_lambda65**.
Изменение к equal3/λ0.75: +0.055 п.п.
Точный состав, веса, источники checkpoints и настройки: selected_candidate.json.

## Ограничения

- Максимум выбран на уже исследуемых данных; возможна подгонка при выборе. Это не ожидаемый hidden-test score.
- mAP — среднее трёх query/gallery эпизодов одной конкретной модели, не среднее нескольких seed.
- Конкретные primary-веса нельзя проверять на alternate: они могли обучаться на его identity.
- Candidate F1/TNR/C — отдельная внутренняя диагностика, не критерий выбора и не конкурсный threshold.
- Head применяется только к своему исходному R1 encoder; в ансамбль head не добавляется.
- Нового обучения/экспорта/релиза нет; selected_candidate.json — описание исследовательского кандидата, не deployment bundle.
- Время без общего лимита, выполненные задачи возобновляются по checksum.
- Защищённые данные/источники сохранены: True.


Точный кандидат: /Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/OSNet-AIN-x1.0/variant_21_model_search/runs/search_v1/selected_candidate.json
Подробные результаты: /Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/OSNet-AIN-x1.0/variant_21_model_search/runs/search_v1/results.json
Это внутренний выбор. Финального threshold, full-train, конкурсного экспорта и смены MVP не было.
